# 🔍 CoreVision — YOLOv8 License Plate Detector Training

**Model:** YOLOv8-nano fine-tuned for license plate detection  
**Dataset:** Roboflow License Plate Recognition v13 (multi-country)  
**Runtime:** T4 GPU (~1-2 hours)

In [ ]:
# CELL 1: Check GPU & Mount Drive
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/CoreVision/weights', exist_ok=True)
print('Drive mounted!')

In [ ]:
# CELL 2: Install Dependencies
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import ultralytics
print('Ultralytics:', ultralytics.__version__)

In [ ]:
# CELL 3: Download Dataset
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")  # <-- paste your key here
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(13)
dataset = version.download("yolov8")

print('Downloaded to:', dataset.location)

In [ ]:
# CELL 4: Find dataset.yaml automatically
import glob, os

# Auto-detect wherever Roboflow downloaded the dataset
yaml_files = glob.glob('/content/**/data.yaml', recursive=True)

if not yaml_files:
    raise FileNotFoundError('data.yaml not found! Make sure Cell 3 ran successfully.')

DATA_YAML = yaml_files[0]
DATASET_PATH = os.path.dirname(DATA_YAML)

print(f'Found data.yaml: {DATA_YAML}')
print(f'Dataset folder: {DATASET_PATH}')

# Count images per split
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    if os.path.exists(img_dir):
        print(f'  {split}: {len(os.listdir(img_dir))} images')

!cat "{DATA_YAML}"

In [ ]:
# CELL 5: Train YOLOv8-nano
from ultralytics import YOLO
import os

DRIVE_WEIGHTS = '/content/drive/MyDrive/CoreVision/weights'

# Resume from Drive checkpoint if it exists
resume_path = f'{DRIVE_WEIGHTS}/plate_detector_last.pt'
if os.path.exists(resume_path):
    print(f'Resuming from checkpoint: {resume_path}')
    model = YOLO(resume_path)
else:
    model = YOLO('yolov8n.pt')

print(f'Training on: {DATA_YAML}')
print('Starting training...')

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    optimizer='AdamW',
    lr0=1e-3,
    weight_decay=5e-4,
    warmup_epochs=3,
    cos_lr=True,
    augment=True,
    mixup=0.1,
    mosaic=1.0,
    degrees=10,
    fliplr=0.5,
    project='/content/runs',
    name='plate_detector',
    save_period=5,
    verbose=True
)

print('Training complete!')

In [ ]:
# CELL 6: Evaluate
best_model_path = '/content/runs/plate_detector/weights/best.pt'

best_model = YOLO(best_model_path)
metrics = best_model.val(data=DATA_YAML, imgsz=640, batch=32)

print('\n📊 Validation Metrics:')
print(f'  mAP@0.5:      {metrics.box.map50:.3f}')
print(f'  mAP@0.5:0.95: {metrics.box.map:.3f}')
print(f'  Precision:    {metrics.box.mp:.3f}')
print(f'  Recall:       {metrics.box.mr:.3f}')

In [ ]:
# CELL 7: Save to Google Drive
import shutil

DRIVE_WEIGHTS = '/content/drive/MyDrive/CoreVision/weights'

shutil.copy('/content/runs/plate_detector/weights/best.pt',
            f'{DRIVE_WEIGHTS}/plate_detector.pt')
shutil.copy('/content/runs/plate_detector/weights/last.pt',
            f'{DRIVE_WEIGHTS}/plate_detector_last.pt')

print('✅ Saved to Google Drive:')
print('  plate_detector.pt       → use this in your project (weights/)')
print('  plate_detector_last.pt  → resume checkpoint')
!ls -lh "{DRIVE_WEIGHTS}/"

In [ ]:
# CELL 8: Visual Test
import glob, matplotlib.pyplot as plt

test_images = glob.glob(f'{DATASET_PATH}/valid/images/*.jpg')[:4]
best_model = YOLO('/content/drive/MyDrive/CoreVision/weights/plate_detector.pt')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, img_path in zip(axes.flat, test_images):
    r = best_model.predict(img_path, conf=0.3, verbose=False)
    ax.imshow(r[0].plot()[:, :, ::-1])
    ax.set_title(f'{len(r[0].boxes)} plates detected')
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CoreVision/detector_samples.png', dpi=150)
plt.show()
print('Samples saved to Drive!')